## Imports && hyperpaarameters (GLOBAL)

In [18]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device='cuda' if torch.cuda.is_available() else 'cpu'
block_size=8
batch_size=4
max_iters=1000
learning_rate=3e-4 # 3^(-4)
eval_iters=250
n_embd=384 
n_head=4
dropout=0.2 # It's gonna dropout random neurons but it is disabled in the eval mode
print(device)

cuda


## Open Dataset

In [8]:
with open('wizard_of_oz.txt','r',encoding='utf-8') as f:
    text=f.read()
print(len(text))
chars=sorted(set(text))
vocab_size=len(chars)
#print(vocab_size)
#print(chars)

232308


## Tokenizers

In [9]:
string_to_int={ch:i for i,ch in enumerate(chars)}
int_to_string={i:ch for i,ch in enumerate(chars)}
encode=lambda s:[string_to_int[c] for c in s]
decode=lambda l:''.join([int_to_string[j] for j in l])

## Get Batch

In [10]:
n=int(0.8*len(data))
train_data=data[:n] 
val_data=data[n:] 
def get_batch(split):
    data=train_data if split=='train'else val_data 
    ix=torch.randint(len(data)-block_size,(batch_size,))
    inputs=torch.stack([data[i:i+block_size] for i in ix])
    targets=torch.stack([data[i+1:i+block_size+1] for i in ix])
    inputs,targets=inputs.to(device),targets.to(device)
    return inputs,targets

## Torch No_Grad

In [11]:
@torch.no_grad()
def estimate_loss():
    out={}
    model.eval()
    for split in ['train','val']:
        losses=torch.zeros(eval_iters) 
        for k in range(eval_iters): 
            X,Y=get_batch(split) 
            logits,loss=model(X,Y) 
            losses[k]=loss.item()
        out[split]=losses.mean()
    model.train()    
    return out         

## Neural Network
In this work we use sine and cosine functions of different frequencies:$$PE_(pos,2i)=sin(\frac{pos}{10000^{2i/d_{model}}})$$ 
$$PE_(pos,2i+1)=cos(\frac{pos}{10000^{2i/d_{model}}})$$
where pos is the position and i is the dimension. That is each dimension of the positional encoding corresponds to a sinusoid(sine wave). The wavelengths form a geometric progression from $2\pi$ to $10000.2\pi$. We choose this function because we hypothesized it would allow the model to easily learn to attend by relative positions, since for any fixed offset $k,PE_{pos+k}$ can be represented as a linear function of $PE_{pos}$.

2. Why PyTorch Uses v · W^T + b Instead of W · v^T + b

1. Row-based memory layout: In Python/NumPy/PyTorch, a batch of N examples is stored as N rows stacked vertically: shape (N, 3).

# Tensor Multiply
When you multiply a 2D tensor (M, K) on the left with a 3D tensor (Batch, K, N) on the right (W @ A), PyTorch broadcasts that single 2D matrix onto the left side of every batch slice in the 3D tensor, producing a 3D output of shape (Batch, M, N).

1. The Shape Rule

• Left 2D Tensor W: (M, K) — One shared M × K matrix.
• Right 3D Tensor A: (Batch, K, N) — Batch separate K × N matrices stacked together.
• Check inner dimensions: K (columns of W) must equal K (rows of each slice A[b]).
• Result C = W @ A: (Batch, M, N), where each batch slice is computed as C[b] = W @ A[b].

2. Concrete By-Hand Example

Let's multiply a 2D matrix W of shape (2, 3) by a 3D tensor A of shape (2, 3, 2) → Result shape: (2, 2, 2).

Left 2D Matrix W — Shape (2, 3):

text
( (1, 0, 2),
  (0, 3, 1) )
Use code with caution.

Right 3D Tensor A — Shape (2, 3, 2) (2 batches of 3 × 2 matrices):

• Batch 0 (A[0]):text
( (1, 2),
  (3, 0),
  (4, 1) )
Use code with caution.
• Batch 1 (A[1]):text
( (2, 1),
  (1, 2),
  (0, 3) )
Use code with caution.

3. Step-by-Step Calculation

1. Multiply W by Batch 0 (W @ A[0]):
	• Row 0, Col 0: (1×1) + (0×3) + (2×4) = 1 + 0 + 8 = 9
	• Row 0, Col 1: (1×2) + (0×0) + (2×1) = 2 + 0 + 2 = 4
	• Row 1, Col 0: (0×1) + (3×3) + (1×4) = 0 + 9 + 4 = 13
	• Row 1, Col 1: (0×2) + (3×0) + (1×1) = 0 + 0 + 1 = 1
	• Batch 0 Result: ( (9, 4), (13, 1) )
2. Multiply the SAME W by Batch 1 (W @ A[1]):
	• Row 0, Col 0: (1×2) + (0×1) + (2×0) = 2 + 0 + 0 = 2
	• Row 0, Col 1: (1×1) + (0×2) + (2×3) = 1 + 0 + 6 = 7
	• Row 1, Col 0: (0×2) + (3×1) + (1×0) = 0 + 3 + 0 = 3
	• Row 1, Col 1: (0×1) + (3×2) + (1×3) = 0 + 6 + 3 = 9
	• Batch 1 Result: ( (2, 7), (3, 9) )

Final 3D Output C — Shape (2, 2, 2):

text
(
  ( ( 9, 4),
    (13, 1) ),

  ( ( 2, 7),
    ( 3, 9) )
)
Use code with caution.

Quick Summary of Both Directions

• 3D (B, M, K) @ 2D (K, N) → (B, M, N): Multiplies every batch slice A[b] on the right by W (A[b] @ W).
• 2D (M, K) @ 3D (B, K, N) → (B, M, N): Multiplies every batch slice A[b] on the left by W (W @ A[b]).
Are you working through custom attention/projection equations where weights sit on the left (W @ X), or exploring PyTorch broadcasting rules in general? Let me know so we can dive into that specific use case.

In [14]:
class Head(nn.Module):
    '''One head of self-attention, but all running in parallel'''
    '''Here we are going to do some scaled hadamar dot products'''
    def __init__(self,,head_size):
        super()__init__()
        '''multihead attention block, so we need to construct K,Q and V'''
        self.key=nn.Linear(n_embed,head_size,bias=False) # Calculating bias will be computationally havy
        self.query=nn.Linear(n_embed,head_size,bias=False) 
        self.value=nn.Linear(n_embed,head_size,bias=False) 
        self.register_buffer()
        '''This is the so called masking layer in the decoder of gtp as opposed to a more complex transformer'''
        self.dropout=nn.Dropout(dropout)
    def forward(self,x): # x is just x
        ''' input x of tensor size=(batch,time-step,channels=n_embd),ndim=3'''
        ''' output out of tensor size=(batch,time-step,channels=n_embd),ndim=3 '''
        B,T,C=x.shape
        k=self.key(x) '''(B,T,hs)''' # It creates W_{384,96}^T but no biases between v=x and z=k; no activation as well
        q=self.query(x) '''(B,T,hs)''' # Same thing
        '''hs means head_size'''
        '''Compute attention score(affinity), 4 reassignments in a row'''
        wei=q@k.transpose(-2,-1)*k.shape[-1]**-0.5  # k.shape[-1] is C or channels=vocab_size
        '''(B,T,hs)@(B,hs,T)=(B,T,T), matmul happens inside the 0th dimensions i.e, batch[0] and batch[1]'''
        wei=wei.masked_fill(self.tril[:T,:T]==0,float('-inf')) 
        '''A triangle low mask boolean with True and False when true represents 0 from tril tensor and makes it -infity '''
        wei=F.softmax(wei,dim=-1) # ReLU on (B,T,T). It adds dimension so one outermoost [] is removed
        '''This is the sigmoid function as opposed to ReLU as we don't want overpowering too much and we get confidence from 0 to 1'''
        '''Here we get the float(-'inf') s to get zeroed out'''
        wei=self.dropout(wei) # Drop out is 20%
        v=self.value(x) # (B,T,hs)
        '''This is the actual web operration of the feed forward on input row vector v'''
        '''preparing for out and it's dimension will have hs in place of channels as opposed to input x'''
        '''So in a nutshell we created even more non linearity and nuances using k,q, a form web activated form of input vector v which gets ftuition in a final matmul'''
        out=wei@v # (B,T,T)@(B,T,hs)=(B,T,hs)
        '''the last phase of SCALED DOT PRODUCT ATTENTION, where input v passing through nn.Linear(v,z) also gets matmuled by it's some more even cooked version of itself vector wei'''
        '''it is like a column vector hs represented in row(more efficient to store on vram) tensors but also carying an extra positional parameter T, the so called parallelism with T=4 per batch'''
        '''Therefore it is a 3d tensor representing (len(out[i])=8 batch members,4 time T members(4 heads?),96 per heads) at (0,1,2) th dimension; extremely easy to deal with as toddler level matrix arithmatics inside thousands of gpu cores'''
        return out
        
class MultiHeadAttention(nn.Module):
    '''multiple heads of self attentions in parallel'''
    def __init__(self,num_heads,head_size): # Different naming probably because of computational overlap error
        super().__init__()
        self.heads=nn.MduleList([Head(head_size) for _ in range(num_heads)]) # I don't know what Head() means, well now he adds class Head() at the top
        '''It gives features for 4 heads in a tensor inside a big ugly tensor [[h_1^{i},...,h_384^{i}],...] where i=1,n=4'''
        self.proj=nn.Linear(head_size*num_hads,n_embd) # We could use kwargs bias=False but we need bias.
        '''this just projects the v of size=(96*4') into z of size=(n_embd,); equal tho, it only has a square weight of size=(384,384)'''
        self.deopout=nn.Dropout(dropout) # For our case 20%, just a smoothie.
    def forward():
        '''So there are __init__() and forward() blocks in all classes, hmmm! Interesting.'''
        out=torch.cat([h(x) for h in self.heads],dim=-1) # Takes away extra sets of []s from the tensors and catenates them into one
        '''This gives [h1,h1,h1,h1,h2,h2,h2,h2,h3,h3,h3,h3,...] unwrapped in a long column vector'''
        out=self.dropout(self.proj(out)) # This is ccconcatenating the four head values into 384=4*96 and everything matches up
        '''So we are just casting the concatenated out tensor as vector size=(384,) through self.proj(x) and getting a vector of size n_embd.'''
        '''The attributes we are not using here will be used outside'''
        return out
        
class FeedForward(nn.Module):
    '''a simple linear layer followed by a non-linearity''' 
    def __init__(self,n_embd):
        super().__init__()
        # We are just creating two layer net with random weights and biases of proper size
        self.net=nn.Sequential(
            nn.Linear(n_embd,4*n_embd), # Linear just unwrap into matrix, a^0_384@w'_(384*4) 
            nn.ReLU(), # the activation, so called nonlinearity
            nn.Linear(4*n_embd,n_embd), # so that internal contact is 4*n_embd and external dimensions n_embd
            nn.Dropout(dropout), # our dropout hyper parameter, smoothens things out even further         
        )
        '''we just construced a sequential parameter to which we are going to plug our input vecor v=x'''
    def forward(self,x): # Class method
        return self.net(x) 
        '''We just plugged in x into the sequential self.net()'''
class Block(nn.Module):
    '''Transformer block: Communication followed by computation'''
    def __init__(self,n_embd,n_head):
        ''' n_embd: embedding dimension(384 options for each input), n_head: the number of heads we'd like'''
        super().__init__() # Inheriting from the init block of nn.Module
        head_size=n_embd//n_head # // is the floor division operator, divides two numbers and rounds the result down to the nearest whole integer
        '''Self explanatory, 384/4=96 features, a_0s? the numbers of featurs each of our heads are going to be capturing'''
        self.sa=MultiHeadAttention(n_head,head_size) # sa short for self attention; 2D tensor param?
        self.ffwd=FeedForward(n_embd) # super nn.Module Feed Forward function
        self.ln1=nn.LayerNorm(n_embd) 
        '''norm based on m=n_embd=384, it helps smoothen out our features a_o^l'''
        self.ln2=nn.LayerNorm(n_embd) # same thing?
        '''All the atrributes constructed'''
    def forward(self,x):
        '''What one decoder layer is doing'''
        y=self.sa(x) # We are unleashing multihead attention on x! 3d?
        x=self.ln1(x+y) # Residual connection, x was from befor and it rewrites x like x=f(x+f(x))
        '''the original architecture, post-norm-> (x+y) first then ln1 slash nn.LayerNorm(n_embed)'''
        y=self.ffwd(x) # We are unleashing multihead attention on rewritten x!
        x=self.ln2(x+y) # We are doing residual connection with rewritten x and rewritting it again, so wriwritten wriritten
        '''He used ln1 and ln2 so that the calling in low level doesnt get contradicted at the same time'''        
        '''ln1 ln2 means the Same but need to be called on different steps by one gpu core, it's the sequential computation unit happening many in parallels'''
        return x
        
class GPTLanguageModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__() # we are inheriting constructor attributes and methods from nn.Module
        self.token_embedding_table=nn.Embedding(vocab_size,n_embd)
        self.position_embedding_table=nn.Embedding(block_size,n_embd)
        self.blocks=nn.Sequential(*[Block(n_embd,n_head=n_head) for _ in range(n_layers)]) 
        '''Above are four stacks of encoders+decoders'''
        self.ln_f=nn.LayerNorm(n_embd)
        '''The so called a_j^L normalization, the final layer norm'''
        self.lm_head=nn.Linear(n_embd,vocab_size)
        '''lm stands for language model head, and this is our linear transformation block at the end of the feed forward'''
        self.apply(self._init_weights) # Any methods or attributes that we deid not define commes from nn.Module.
    def _init_weights(self,module):
        if isinstance(module,nn.Linear):
            torch.nn.init.normal_(module.weight,mean=0.0,std=0.02) # std is standard deviation
            '''Probably some random weight matrix generation from module.weight based on our nn.Sequential'''
            if module.bias is not none:
                torch.nn.init.zeros_(module.bias)
                '''Probably some special treatment for module.bias'''
        elif isinstance(module,nn.Embedding):
            torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)
            '''Same thing as if except the bias elimination, probably we do not need bias at first'''
    def forward(self,index,targets=None):
        logits=self.token_embedding_table(index)
        # indexes and targets are both (B,T) tensor of integers
        tok_embd=self.token_embedding_table(idx) # (B,T,C) same as logits
        pos_embd=self.position_embedding_table(torch_arange(T,device=device)) # (T,C), it's going parallel
        x=tok_embd+pos_embd # (B,T,C) same as logits, broadcasted, so hadamar element wize rule
        '''we just prepared our positional embedding and we pass them directly to our sequential below'''
        x=self.blocks(x) # nn.Sequential applied
        x=self.ln_f() # final layer norm
        logits=self.lm_head(x) # Linear so matmul happened; size=(B,T,vocab_size)
        '''Is logits reassigned here?'''
        '''ready for softmax- to get it to probabilities that we can feed into our softmax'''
        if targets is None:
            loss=None
        else:
            B,T,C=logits.shape
            logits=logits.view(B*T,C)
            targets=targets.view(B*T)
            loss=F.cross_entropy(logits,targets)
        return logits,loss        
    def generate(self,index,max_new_tokens):
        for _ in range(max_new_tokens):
            logits,_=self.forward(index)
            logits=logits[:,-1,:]
            probs=F.softmax(logits,dim=-1)
            index_next=torch.multinomial(probs, num_samples=1)
            index=torch.cat((index,index_next),dim=1)
        return index        
model=GPTLanguageModel(vocab_size).to(device)

## Optimizer

In [15]:
optimizer=torch.optim.AdamW(model.parameters(),lr=learning_rate)
for iter in range(max_iters):
    if iter%eval_iters==0:
        losses=estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")
    xb,yb=get_batch('train')
    _,loss=model.forward(xb,yb)
    optimizer.zero_grad(set_to_none=True) # It means we are also excluding 0. which takes place and set zer_grad to None instead of 0. float
    loss.backward()
    optimizer.step()
#loss.item()

step: 0, train loss: 4.877, val loss: 4.842
step: 250, train loss: 4.820, val loss: 4.776
step: 500, train loss: 4.765, val loss: 4.737
step: 750, train loss: 4.680, val loss: 4.668


4.783543586730957